In [ ]:
import os
import sys
from pathlib import Path

# Run this notebook from the repo root (audio-clf)
REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

os.environ["DATASETS_AUDIO_BACKEND"] = "soundfile"
os.environ["TORCHCODEC_QUIET"] = "1"

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

from load_data import load

dataset = load()

# Pick the held-out split the model never trained on
TEST_SPLIT = next(
    (s for s in ("test", "validation", "val") if s in dataset),
    list(dataset.keys())[-1]   # fallback: last split
)

print("Splits:", list(dataset.keys()))
for k in dataset.keys():
    print(f"  {k}: {len(dataset[k])} rows, columns: {dataset[k].column_names}")
print(f"\nUsing '{TEST_SPLIT}' as the held-out test split.")


In [ ]:
from inference import resolve_checkpoint, load_model, run_row_inference
from load_data import get_row, read_audio
from train import SAMPLE_RATE
import numpy as np
from IPython.display import display, Audio

ckpt_path = resolve_checkpoint()
print(f"Using checkpoint: {ckpt_path}")

model, emotion_encoder, gender_encoder, age_encoder, \
    id2emotion, id2gender, id2age, processor = load_model(ckpt_path)
print("Model ready.")


In [ ]:
def check_row(index: int, split: str | None = None):
    split = split or TEST_SPLIT
    row = get_row(split, index, dataset=dataset)

    audio_data, sample_rate = read_audio(row["audio"])
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)
    audio_data = audio_data.astype(np.float32)
    duration_sec = len(audio_data) / sample_rate
    audio_widget = Audio(data=audio_data, rate=sample_rate, normalize=True, embed=True)

    # ── Inference ──────────────────────────────────────────────────────────────
    results = run_row_inference(row, model, processor, id2emotion, id2gender, id2age)

    # ── Print summary ──────────────────────────────────────────────────────────
    sep = "─" * 56
    print(sep)
    print(f"  Row {index}  |  split: '{split}'  |  {duration_sec:.2f}s")
    transcript = row.get("transcribed_text") or row.get("text") or row.get("transcript") or "—"
    description = row.get("text_description", "")
    print(f"  Transcript : {transcript}")
    if description:
        print(f"  Description: {description}")
    print(sep)
    for task, gt_key in [("emotion", "emotion"), ("gender", "gender"), ("age", "age_category")]:
        r = results[task]
        gt = row.get(gt_key, "—")
        match = "✓" if r["pred"] == str(gt) else "✗"
        print(f"\n  {task.capitalize():<10} pred: {r['pred']} ({r['confidence']*100:.1f}%)  {match}"
              f"   gt: {gt}")
        print(f"             top-3: {[(lbl, f'{p*100:.1f}%') for lbl, p in r['top3']]}")
    print(f"\n{sep}")

    display(audio_widget)
    return audio_widget, results


In [ ]:
# ── Change index to inspect any row from the test split ──
audio, results = check_row(index=151)

In [ ]:
# 83, 30, 32, 33, 53

In [ ]:
# Rows with ground-truth emotion = sad
split = dataset[TEST_SPLIT]
sad_indices = [i for i in range(len(split)) if split[i]["emotion"] == "sad"]
print(f"Rows with emotion gt = 'sad': {len(sad_indices)}")
for i in sad_indices:
    row = split[i]
    transcript = (row.get("transcribed_text") or row.get("text") or "")[:70]
    print(f"  {i}: gender={row.get('gender')}, age={row.get('age_category')} | {transcript}...")